In [1]:
import cobra
import os
import pickle
from cobra.flux_analysis import (
    single_gene_deletion)
import re
from cobra.manipulation.delete import remove_genes
from cobra.manipulation import rename_genes
human_gem= cobra.io.read_sbml_model('/data/user_home/dlwnsrb/cow_reconstruction/Human-GEM.xml')

with open('biblastp_match.pickle', 'rb') as f:
    biblastp_match= pickle.load(f)

In [2]:
ensg_list= list()
for gene in human_gem.genes:
    if 'ENSG' in gene.id:
        ensg_list.append(gene)
df= single_gene_deletion(human_gem,ensg_list)
essential_gene_list= []
optimize_value= human_gem.optimize().objective_value
for index, row in df[['ids','growth']].iterrows():
    if optimize_value - row['growth'] > 0.5:
        essential_gene_list.append(list(row['ids'])[0])
rename_genes(human_gem, biblastp_match)

In [6]:
remove_gene_list=[]
for gene in human_gem.genes:
    if 'ENSG' in gene.id and not (gene.id in essential_gene_list):
        remove_gene_list.append(gene.id)
remove_genes(human_gem, remove_gene_list)

/data/user_home/dlwnsrb/anaconda3/envs/cobra/lib/python3.8/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


In [14]:
from cobra.io import write_sbml_model
write_sbml_model(human_gem, "change_gene_name_cow.xml")